# Erasus — Video Model Unlearning (VideoMAE-base, Tesla T4)

This notebook demonstrates **coreset-based unlearning** on a video foundation model.
We fine-tune [VideoMAE-base](https://huggingface.co/MCG-NJU/videomae-base) (87M params)
as an action classifier on synthetic video frames, then use
[Erasus](https://github.com/OnePunchMonk/erasus) to surgically remove specific
action classes via **gradient ascent** while varying:

- **Coreset selector** — 4 methods (`gradient_norm`, `el2n`, `herding`, `random`)
- **Coreset fraction** — 5 values (5%, 10%, 20%, 50%, 100% of the forget set)

VideoMAE expects input tensors of shape `[B, 16, 3, 224, 224]` (16 frames per clip).
We use small sample counts (100 forget, 300 retain) and batch size 2 to stay within
the 15 GB VRAM budget of a free Colab T4 GPU.

**Runtime:** ~20–35 min on a free Colab T4 GPU.

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib transformers
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import gc
import time
import json
import math
import warnings
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import VideoMAEModel

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------------------------------------------ #
# Model
# ------------------------------------------------------------------ #

class VideoMAEClassifier(nn.Module):
    """VideoMAE encoder + linear classification head."""

    def __init__(self, videomae, n_classes=10):
        super().__init__()
        self.encoder = videomae
        self.classifier = nn.Linear(768, n_classes)  # videomae-base hidden=768

    def forward(self, x, labels=None, **kwargs):
        # x: [B, num_frames, C, H, W]
        out = self.encoder(pixel_values=x).last_hidden_state  # [B, T, 768]
        pooled = out.mean(dim=1)  # [B, 768]
        logits = self.classifier(pooled)  # [B, n_classes]
        if labels is not None:
            return type("Out", (), {
                "logits": logits,
                "loss": F.cross_entropy(logits, labels),
            })()
        return logits

# ------------------------------------------------------------------ #
# Helpers
# ------------------------------------------------------------------ #

def compute_accuracy(model, loader, device):
    """Return accuracy (0-1) of *model* on *loader*."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            videos, labels = batch[0].to(device), batch[1].to(device)
            logits = model(videos)
            if hasattr(logits, "logits"):
                logits = logits.logits
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

# ------------------------------------------------------------------ #
# Synthetic video data
# ------------------------------------------------------------------ #
# Each class gets a unique sinusoidal temporal pattern baked into the
# frames so the model can learn class-discriminative features.

N_CLASSES = 10
FORGET_CLASSES = [0, 1]
RETAIN_CLASSES = list(range(2, N_CLASSES))
NUM_FRAMES = 16
IMG_SIZE = 224
N_FORGET = 100   # samples with forget-class labels
N_RETAIN = 300   # samples with retain-class labels

def make_synthetic_videos(n_samples, classes, seed=42):
    """Generate synthetic video clips with class-specific temporal patterns.

    Returns (videos: [N, 16, 3, 224, 224], labels: [N]) as float32/long tensors.
    Videos are generated one at a time to avoid OOM during construction.
    """
    rng = torch.Generator().manual_seed(seed)
    n_per_class = n_samples // len(classes)
    videos_list = []
    labels_list = []

    for cls in classes:
        for _ in range(n_per_class):
            # Base noise
            vid = torch.randn(NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE, generator=rng) * 0.1
            # Class-specific sinusoidal temporal pattern
            freq = (cls + 1) * 0.5
            t = torch.linspace(0, 2 * math.pi, NUM_FRAMES)
            signal = torch.sin(freq * t)  # [16]
            vid += signal.view(NUM_FRAMES, 1, 1, 1) * 0.5
            # Class-specific spatial stripe
            stripe_start = (cls * IMG_SIZE // N_CLASSES)
            stripe_end = stripe_start + IMG_SIZE // N_CLASSES
            vid[:, cls % 3, stripe_start:stripe_end, :] += 1.0
            videos_list.append(vid.unsqueeze(0))  # [1, 16, 3, 224, 224]
            labels_list.append(cls)

    videos = torch.cat(videos_list, dim=0)  # [N, 16, 3, 224, 224]
    labels = torch.tensor(labels_list, dtype=torch.long)
    return videos, labels

print("Generating synthetic forget videos ...")
forget_videos, forget_labels = make_synthetic_videos(N_FORGET, FORGET_CLASSES, seed=42)
print(f"  Forget set: {forget_videos.shape}, labels: {forget_labels.shape}")

print("Generating synthetic retain videos ...")
retain_videos, retain_labels = make_synthetic_videos(N_RETAIN, RETAIN_CLASSES, seed=123)
print(f"  Retain set: {retain_videos.shape}, labels: {retain_labels.shape}")

forget_dataset = TensorDataset(forget_videos, forget_labels)
retain_dataset = TensorDataset(retain_videos, retain_labels)
train_dataset = TensorDataset(
    torch.cat([forget_videos, retain_videos], dim=0),
    torch.cat([forget_labels, retain_labels], dim=0),
)

forget_loader = DataLoader(forget_dataset, batch_size=2, shuffle=True)
retain_loader = DataLoader(retain_dataset, batch_size=2, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# ------------------------------------------------------------------ #
# Load VideoMAE-base and build classifier
# ------------------------------------------------------------------ #

print("\nLoading VideoMAE-base from HuggingFace ...")
videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEClassifier(videomae, n_classes=N_CLASSES).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"VideoMAEClassifier: {n_params / 1e6:.1f}M parameters")

# ------------------------------------------------------------------ #
# Train (fine-tune) for 5 epochs
# ------------------------------------------------------------------ #

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("\nFine-tuning for 5 epochs ...")
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        videos, labels = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        out = model(videos, labels=labels)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"  Epoch {epoch} | loss {avg_loss:.4f}")

base_forget_acc = compute_accuracy(model, forget_loader, device)
base_retain_acc = compute_accuracy(model, retain_loader, device)
print(f"\nBase model  \u2014  Forget acc: {base_forget_acc:.4f} | Retain acc: {base_retain_acc:.4f}")

# Snapshot trained weights
trained_state = deepcopy(model.state_dict())
print("Saved trained state.")

In [ ]:
from erasus import ErasusUnlearner

# ------------------------------------------------------------------ #
# Coreset ablation
# ------------------------------------------------------------------ #

FRACTIONS = [0.05, 0.1, 0.2, 0.5, 1.0]
SELECTORS = ["gradient_norm", "el2n", "herding", "random"]

results = {}  # {selector: {fraction: {forget_acc, retain_acc, time_s}}}

total_runs = len(SELECTORS) * len(FRACTIONS)
run_idx = 0

for selector_name in SELECTORS:
    results[selector_name] = {}
    for frac in FRACTIONS:
        run_idx += 1
        tag = f"[{run_idx}/{total_runs}]"

        # Fresh copy of the trained model
        m = VideoMAEClassifier(
            VideoMAEModel.from_pretrained("MCG-NJU/videomae-base"),
            n_classes=N_CLASSES,
        ).to(device)
        m.load_state_dict(deepcopy(trained_state))

        try:
            unlearner = ErasusUnlearner(
                model=m,
                strategy="gradient_ascent",
                selector=selector_name,
                selector_config={"fraction": frac},
            )
            t0 = time.time()
            unlearner.fit(
                forget_data=forget_loader,
                retain_data=retain_loader,
                epochs=5,
            )
            elapsed = time.time() - t0

            f_acc = compute_accuracy(m, forget_loader, device)
            r_acc = compute_accuracy(m, retain_loader, device)
        except Exception as exc:
            print(f"{tag}  {selector_name:>15s}  frac={frac:.2f}  \u2014  ERROR: {exc}")
            f_acc = float("nan")
            r_acc = float("nan")
            elapsed = 0.0

        results[selector_name][frac] = {
            "forget_acc": f_acc,
            "retain_acc": r_acc,
            "time_s": round(elapsed, 2),
        }
        print(
            f"{tag}  {selector_name:>15s}  frac={frac:.2f}  |  "
            f"forget={f_acc:.4f}  retain={r_acc:.4f}  ({elapsed:.1f}s)"
        )

        # Free GPU memory between runs
        del m, unlearner
        gc.collect()
        torch.cuda.empty_cache()

# ------------------------------------------------------------------ #
# Summary table
# ------------------------------------------------------------------ #

print("\n" + "=" * 80)
print(f"{'Selector':>15s}  {'Frac':>5s}  {'Forget':>8s}  {'Retain':>8s}  {'Time(s)':>8s}")
print("-" * 80)
for sel in SELECTORS:
    for frac in FRACTIONS:
        r = results[sel][frac]
        print(
            f"{sel:>15s}  {frac:>5.2f}  "
            f"{r['forget_acc']:>8.4f}  {r['retain_acc']:>8.4f}  {r['time_s']:>8.2f}"
        )
    print()
print("=" * 80)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------------ #
# Prepare arrays
# ------------------------------------------------------------------ #

fracs = np.array(FRACTIONS)

forget_curves = {}
retain_curves = {}
trade_curves = {}

for sel in SELECTORS:
    fa = np.array([results[sel][f]["forget_acc"] for f in FRACTIONS])
    ra = np.array([results[sel][f]["retain_acc"] for f in FRACTIONS])
    forget_curves[sel] = fa
    retain_curves[sel] = ra
    trade_curves[sel] = (1.0 - fa) * ra  # higher = better

# ------------------------------------------------------------------ #
# 3-panel figure
# ------------------------------------------------------------------ #

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for sel in SELECTORS:
    axes[0].plot(fracs, forget_curves[sel], marker="o", label=sel)
    axes[1].plot(fracs, retain_curves[sel], marker="o", label=sel)
    axes[2].plot(fracs, trade_curves[sel], marker="o", label=sel)

axes[0].set_title("Forget Accuracy vs Coreset Fraction")
axes[0].set_xlabel("Coreset Fraction")
axes[0].set_ylabel("Forget Accuracy")
axes[0].axhline(base_forget_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Retain Accuracy vs Coreset Fraction")
axes[1].set_xlabel("Coreset Fraction")
axes[1].set_ylabel("Retain Accuracy")
axes[1].axhline(base_retain_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Tradeoff Score  (1 - forget) * retain")
axes[2].set_xlabel("Coreset Fraction")
axes[2].set_ylabel("Tradeoff Score")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

fig.suptitle("VideoMAE-base Coreset Unlearning Ablation (Tesla T4)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig("/content/video_coreset_ablation_3panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/video_coreset_ablation_3panel.png")

In [ ]:
import numpy as np

# ------------------------------------------------------------------ #
# Save results JSON and download
# ------------------------------------------------------------------ #

output = {
    "experiment": "video_videomae_coreset_ablation",
    "model": "VideoMAEClassifier (MCG-NJU/videomae-base)",
    "model_params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
    "dataset": "synthetic_video (16 frames x 3 x 224 x 224)",
    "n_classes": N_CLASSES,
    "forget_classes": FORGET_CLASSES,
    "retain_classes": RETAIN_CLASSES,
    "forget_samples": N_FORGET,
    "retain_samples": N_RETAIN,
    "train_epochs": 5,
    "unlearn_epochs": 5,
    "strategy": "gradient_ascent",
    "base_forget_acc": base_forget_acc,
    "base_retain_acc": base_retain_acc,
    "fractions": FRACTIONS,
    "selectors": SELECTORS,
    "results": {},
}

for sel in SELECTORS:
    output["results"][sel] = {}
    for frac in FRACTIONS:
        r = results[sel][frac]
        output["results"][sel][str(frac)] = {
            "forget_acc": round(r["forget_acc"], 6) if not np.isnan(r["forget_acc"]) else None,
            "retain_acc": round(r["retain_acc"], 6) if not np.isnan(r["retain_acc"]) else None,
            "time_s": r["time_s"],
        }

json_path = "/content/video_coreset_ablation_results.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {json_path}")
print(json.dumps(output, indent=2))

# Download files (Colab-specific)
try:
    from google.colab import files
    files.download(json_path)
    files.download("/content/video_coreset_ablation_3panel.png")
except ImportError:
    print("Not running on Colab \u2014 files saved locally under /content/")